# 01 - Clean Salon Appointment Log

**Input:** data/salon/appointments_anoymized.csv (raw, never modified)  

**Output:** data/processed/salon_clean.csv  

Steps follow Task 2 sections D4 and D5a.  

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

# Notebook lives in notebooks/, so go up one folder to reach data/
RAW_PATH = "../data/salon/appointments_anonymized.csv"

df = pd.read_csv(RAW_PATH)

# Stop the notebook if the raw file is not what D4 describes
assert df.shape == (186, 12), f"Unexpected shape: {df.shape}"

print(df.shape)
print(list(df.columns))
df.head()

(186, 12)
['appointment_hash', 'client_pid', 'provider_pid', 'services', 'booking_method', 'status', 'deposit_amount', 'deposit_collected', 'charge_hash', 'appt_datetime_utc', 'booked_datetime_utc', 'card_exp_year']


,appointment_hash,client_pid,provider_pid,services,booking_method,status,deposit_amount,deposit_collected,charge_hash,appt_datetime_utc,booked_datetime_utc,card_exp_year
0,e65a69f03395cf1714976bcded41418571003014a676fd...,Client_105,Provider_001,Women's Haircut (Add on deep conditioning ),Self-Scheduled,cancelled,0,False,NaN,2025-02-13T10:00:00Z,2025-02-11T18:38:00Z,2028.0
1,a1df894d96675bdd23df9a701ffcb0b3aac44f6ad674c3...,Client_064,Provider_001,Women's Haircut,Self-Scheduled,completed,0,False,1eaaf197db40adee2a37bb689a09a6cd126fab9091916b...,2025-02-27T09:00:00Z,2025-02-05T12:55:00Z,NaN
2,fe5da58adef34c224e34699e0ae91dad58ff4c310d95bf...,Client_022,Provider_001,All over color,Self-Scheduled,completed,0,False,8518a65435c1bbeb016e1d2fe36ab217045a9cfb42cd8b...,2025-03-17T15:00:00Z,2025-03-03T11:26:00Z,NaN
3,16640665331a36c8b45a35b27aa846b71050c1e829be01...,Client_079,Provider_001,Balayage/lived in/Foilayage,Self-Scheduled,completed,0,False,5eb22f2454c8a1cf335e75eee280bc9a574fadb10d9b2a...,2025-03-26T14:30:00Z,2025-03-07T17:50:00Z,NaN
4,b2b4633d7294fb0785bf4cf899f0071c478157dfb2dee1...,Client_084,Provider_001,"Partial Custom Highlight, Add on base color",Self-Scheduled,completed,0,False,df9f604d703e0268a8f86885eb0c64f6b3639f1187514c...,2025-03-27T12:00:00Z,2025-03-09T12:01:00Z,NaN


In [2]:
# D4 says every deposit is zero - here shows it before dropping
print("deposit_amount values:\n", df["deposit_amount"].value_counts(dropna=False))

# Find both deposit columns by name instead of guessing
deposit_cols = [c for c in df.columns if c.startswith("deposit")]
print("Deposit columns found:", deposit_cols)

# D4/ D5a: payment fields and Client ID are not used in analysis
drop_cols = ["charge_hash", "card_exp_year", "client_pid"] + deposit_cols
df = df.drop(columns=drop_cols)

print("Columns remaining:", df.shape[1])
print(list(df.columns))

deposit_amount values:
 deposit_amount
0    186
Name: count, dtype: int64
Deposit columns found: ['deposit_amount', 'deposit_collected']
Columns remaining: 7
['appointment_hash', 'provider_pid', 'services', 'booking_method', 'status', 'appt_datetime_utc', 'booked_datetime_utc']


In [3]:
# D4 validity check: every appointmnet should be unique
n_dupes = df["appointment_hash"].duplicated().sum()
print("Duplicate appointment_hash values:", n_dupes)
assert n_dupes == 0, "Duplicates found - investigate before continuing"

# D5a: hash is kept only for duplicate check, then removed
df = df.drop(columns=["appointment_hash"])
print("Columns remaining:", df.shape[1])

Duplicate appointment_hash values: 0
Columns remaining: 6


In [4]:
# utc=True reads the timestamps as UTC (as labeled in file)
# tz_localize(None) then removes the label without shifting hour,
# so 14:00 stays 14:00. D4: times are treated as local salon time
df["appt_dt"] = pd.to_datetime(df["appt_datetime_utc"], utc=True).dt.tz_localize(None)
df["booked_dt"] = pd.to_datetime(df["booked_datetime_utc"], utc=True).dt.tz_localize(None)

# Raw text columns removed as not needed
df = df.drop(columns=["appt_datetime_utc", "booked_datetime_utc"])

# CHeck against profiling: 182 in 9 - 17, one at 8, two at 18 and one at 1
print(df["appt_dt"].dt.hour.value_counts().sort_index())

appt_dt
1      1
8      1
9     30
10    23
11    24
12    24
13    18
14    18
15    16
16    19
17    10
18     2
Name: count, dtype: int64


In [5]:
# Hour of the appointment - basis for demand tiers (C1)
df["appt_hour"] = df["appt_dt"].dt.hour

# Days between booking and appointment (lead time)
df["lead_days"] = (df["appt_dt"] - df["booked_dt"]).dt.days

# Week number counted from the first appointment in the log.
# Built from the data (not calendar weeks) due to per-client
# date shift (D4). //Using whole num division
first_appt = df["appt_dt"].min()
df["week_index"] = (df["appt_dt"] - first_appt).dt.days // 7

# Check against D4
print("Median lead days:", df["lead_days"].median()) # expecting 14
print("Negative lead times:", (df["lead_days"] < 0).sum()) # expecting 2
print("week_index range:", df["week_index"].min(), "to", df["week_index"].max()) # expecting 0 - 43

df.head()

Median lead days: 14.0
Negative lead times: 2
week_index range: 0 to 43


,provider_pid,services,booking_method,status,appt_dt,booked_dt,appt_hour,lead_days,week_index
0,Provider_001,Women's Haircut (Add on deep conditioning ),Self-Scheduled,cancelled,2025-02-13 10:00:00,2025-02-11 18:38:00,10,1,0
1,Provider_001,Women's Haircut,Self-Scheduled,completed,2025-02-27 09:00:00,2025-02-05 12:55:00,9,21,1
2,Provider_001,All over color,Self-Scheduled,completed,2025-03-17 15:00:00,2025-03-03 11:26:00,15,14,4
3,Provider_001,Balayage/lived in/Foilayage,Self-Scheduled,completed,2025-03-26 14:30:00,2025-03-07 17:50:00,14,18,5
4,Provider_001,"Partial Custom Highlight, Add on base color",Self-Scheduled,completed,2025-03-27 12:00:00,2025-03-09 12:01:00,12,17,6


In [6]:
# --- Hour handling (D4) ---
# Keeping  raw hour for the record
df["appt_hour_raw"] = df["appt_hour"]

# 8 and 18 are one hour outside the 9-17 booking day:
# snap them to the nearest slot so capacity stays at 9 slots (D4)
near_edge = df["appt_hour_raw"].isin([8, 18])
print("Snapped to 9 or 17:", near_edge.sum())          # expecting 3

# 1 am is not plausible salon time (probable entry error). Keep the row
# for weekly counts, lead time, status and clustering, but exclude it
# from every hour-of-day analysis (tiers, hour chart, and Kruskal-Wallis).
df["hour_valid"] = df["appt_hour_raw"].between(8, 18)
print("Excluded from hour analysis:", (~df["hour_valid"]).sum())  # expecting 1

df["appt_hour"] = df["appt_hour_raw"].clip(lower=9, upper=17)
df.loc[~df["hour_valid"], "appt_hour"] = pd.NA

# B6: no more than 5% of records removed or excluded
excluded = (~df["hour_valid"]).sum()
print(f"Share excluded: {excluded / len(df):.1%}")      # expecting 0.5%
assert excluded / len(df) <= 0.05

print(df.loc[df["hour_valid"], "appt_hour"].value_counts().sort_index())
# expect 9: 31, 10-16 unchanged, 17: 12

Snapped to 9 or 17: 3
Excluded from hour analysis: 1
Share excluded: 0.5%
appt_hour
9.0     31
10.0    23
11.0    24
12.0    24
13.0    18
14.0    18
15.0    16
16.0    19
17.0    12
Name: count, dtype: int64


In [7]:
# D4: inspecting raw service names before standardizing - 39 expected
print("Unique raw names:", df["services"].str.strip().nunique())
print(df["services"].str.strip().value_counts().to_string())

Unique raw names: 39
services
Women's Haircut                                                     40
Balayage/lived in/Foilayage                                         17
All over color                                                      15
Signature Blonde Session                                            14
Gloss with haircut                                                  11
Balayage/Foilayage partial                                          11
Full custom Highlight                                                8
Partial Custom Highlight                                             8
Partial Custom Highlight, Add on base color                          7
Root touch-up with cut                                               6
Free Consultation|Color Correction|                                  5
Mini Custom Highlight, Add on base color                             5
Blowout/styling                                                      4
Womens Dry Cut                                 

In [8]:
import re

def normalize(text):
    """lowercase, straighten apostrophes, pipes to spaces, collapse spaces"""
    text = text.lower().replace("\u2019", "'").replace("'", "").replace("|", " ")
    return re.sub(r"\s+", " ", text).strip()

def split_services(raw):
    """split one entry on , ( ) into cleaned pieces"""
    parts = re.split(r"[,()]", raw)
    return [normalize(p) for p in parts if normalize(p)]

# Pieces that are add-ons, never the main service
ADDONS = {"add on base color", "extra color bowls", "add on deep conditioning",
          "deep conditioning treatment"}

# Small services: main service only if nothing bigger is in the same booking
MINOR = {"eye brow wax only", "wax", "gloss"}

# Normalized piece -> standard service name
SERVICE_MAP = {
    "womens haircut":              "Women's Haircut",
    "womens dry cut":              "Women's Dry Cut",
    "balayage/lived in/foilayage": "Balayage (Full)",
    "balayage/foilayage partial":  "Balayage (Partial)",
    "all over color":              "All Over Color",
    "signature blonde session":    "Signature Blonde Session",
    "gloss with haircut":          "Gloss with Haircut",
    "full custom highlight":       "Full Custom Highlight",
    "partial custom highlight":    "Partial Custom Highlight",
    "mini custom highlight":       "Mini Custom Highlight",
    "root touch-up with cut":      "Root Touch-Up with Cut",
    "root touch-up only":          "Root Touch-Up Only",
    "free consultation color correction": "Consultation",
    "color corrections/vivid colors": "Color Correction",
    "blowout/styling":             "Blowout/Styling",
    "2 row extension move up":     "Extension Move-Up",
    "eye brow wax only":           "Brow Wax",
    "wax":                         "Wax",
        "1 row extension move up":     "Extension Move-Up",
    "gloss":                       "Gloss",
    "deep conditioning treatment": "Deep Conditioning",
}

# --- Check: every piece must be either an add-on or in SERVICE_MAP ---
all_pieces = {p for raw in df["services"] for p in split_services(raw)}
unmapped = sorted(all_pieces - ADDONS - set(SERVICE_MAP))
print("Unmapped pieces:", unmapped)
assert not unmapped, "Add these to SERVICE_MAP or ADDONS, then rerun"

Unmapped pieces: []


In [9]:
def primary_service(raw):
    pieces = [p for p in split_services(raw) if p not in ADDONS]
    major = [p for p in pieces if p not in MINOR]
    main = (major or pieces or split_services(raw))[0]
    return SERVICE_MAP.get(main, main)

df["service"] = df["services"].apply(primary_service)

# Anything booked alongside the main service (add-on or second service)
df["has_addon"] = df["services"].apply(lambda r: len(split_services(r)) > 1)

print("Raw names:", df["services"].str.strip().nunique(), "-> primary services:", df["service"].nunique())
print(df["service"].value_counts().to_string())
print("\nBookings with an add-on:", df["has_addon"].sum())

# C4: services with fewer than 5 bookings grouped as "Other" (clustering only)
counts = df["service"].value_counts()
df["service_group"] = df["service"].where(df["service"].map(counts) >= 5, "Other")
print("\nClustering groups:\n", df["service_group"].value_counts().to_string())

# D4: inspect raw service names before standardizing (39 expected)
print("Unique raw names:", df["services"].str.strip().nunique())
print(df["services"].str.strip().value_counts().to_string())

# D4 evidence: raw service names before standardization
df["services"].str.strip().value_counts().to_csv("../outputs/tables/raw_service_names.csv")
print("Saved raw_service_names.csv")

Raw names: 39 -> primary services: 18
service
Women's Haircut             42
Partial Custom Highlight    20
Balayage (Full)             18
Balayage (Partial)          16
All Over Color              15
Signature Blonde Session    14
Full Custom Highlight       13
Gloss with Haircut          11
Mini Custom Highlight        8
Root Touch-Up with Cut       6
Consultation                 5
Blowout/Styling              4
Women's Dry Cut              4
Extension Move-Up            3
Root Touch-Up Only           3
Gloss                        2
Wax                          1
Color Correction             1

Bookings with an add-on: 34

Clustering groups:
 service_group
Women's Haircut             42
Partial Custom Highlight    20
Balayage (Full)             18
Other                       18
Balayage (Partial)          16
All Over Color              15
Signature Blonde Session    14
Full Custom Highlight       13
Gloss with Haircut          11
Mini Custom Highlight        8
Root Touch-Up with Cut

In [10]:
# --- Status (D4): checkout counts as completed ---
df["status"] = df["status"].str.strip().str.lower().replace({"checkout": "completed"})
df["cancel_or_noshow"] = df["status"].isin(["cancelled", "no_show"]).astype(int)
print(df["status"].value_counts())
print("Cancel/no-show rate:", round(df["cancel_or_noshow"].mean(), 3))   # expect 0.134

status
completed    161
cancelled     22
no_show        3
Name: count, dtype: int64
Cancel/no-show rate: 0.134


In [11]:
# --- Lead time (D4): negative lead times excluded from lead-time analysis only ---
df["lead_valid"] = df["lead_days"] >= 0
print("Negative lead times:", (~df["lead_valid"]).sum())                 # expect 2
df.loc[~df["lead_valid"], "lead_days"] = pd.NA
print("Lead-time quartiles:", df["lead_days"].quantile([.25, .5, .75]).tolist())  # ~[4, 14, 44]

Negative lead times: 2
Lead-time quartiles: [4.0, 14.5, 44.25]


In [12]:
# Save cleaned table
import os
os.makedirs("../data/processed", exist_ok=True)

keep = ["provider_pid", "services", "service", "service_group", "has_addon",
        "booking_method", "status", "cancel_or_noshow",
        "appt_dt", "booked_dt", "appt_hour_raw", "appt_hour", "hour_valid",
        "lead_days", "lead_valid", "week_index"]
clean = df[keep]
assert len(clean) == 186
clean.to_csv("../data/processed/salon_clean.csv", index=False)
print("Saved salon_clean.csv:", clean.shape)

Saved salon_clean.csv: (186, 16)
